In [22]:
import torch
import torch.nn as nn

class MHApytorchScaledProduct(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = dropout

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        qkv = self.qkv(x)
        qkv = qkv.view(b, num_tokens, 3, self.num_heads, self.head_dim)
        # (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv
        use_dropout = 0 if not self.training else self.dropout
        context_vector = nn.functional.scaled_dot_product_attention(query, key, value, attn_mask=None, dropout_p=use_dropout, is_causal=True)
        context_vector = context_vector.transpose(1,2).contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector

In [23]:
batch_size = 8
context_length = 1024
embed_dim = 768
device = "cuda" if torch.cuda.is_available() else "cpu"
input = torch.randn((batch_size, context_length, embed_dim), device=device)

In [24]:
mhspa = MHApytorchScaledProduct(
    d_in=embed_dim,
    d_out=embed_dim,
    num_heads=12, 
    context_length=context_length,
    qkv_bias=False
).to(device)

out = mhspa(input)
print(out.shape)

torch.Size([8, 1024, 768])


In [25]:
%timeit mhspa(input)

257 ms ± 41.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
